# Task 4

In [ ]:
import os
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim.models import Word2Vec
import nltk
import logging


# Seting up logging-
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


# setting NLTK Data Path and Download Required Data
# setting NLTK data path
nltk_data_path = os.path.join(
    os.path.expanduser("~"), "AppData", "Roaming", "nltk_data"
)
os.makedirs(nltk_data_path, exist_ok=True)
nltk.data.path.append(nltk_data_path)

# doownloading all required NLTK data
required_nltk_data = [
    "punkt",
    "averaged_perceptron_tagger",
    "maxent_ne_chunker",
    "words",
]
for item in required_nltk_data:
    try:
        nltk.download(item, quiet=True)
    except Exception as e:
        logging.error(f"Error downloading {item}: {str(e)}")


#loading the Data

try:
    data_path = r"D:\YearTwoAI\Block C\2024-25c-fai2-adsai-VictoriaVicheva233182\Week 1\transcribed_data_assemblyAI.csv"
    df = pd.read_csv(data_path)

    if "Sentence" not in df.columns:
        raise ValueError("The CSV does not contain a column named 'Sentence'.")
    logging.info("Data loaded successfully")
except Exception as e:
    logging.error(f"Error loading data: {str(e)}")
    raise


# (POS) Tagging
try:
    nlp = spacy.load("en_core_web_sm")
except Exception as e:
    logging.error(f"Error loading spaCy model: {str(e)}")
    raise


def extract_pos_tags(sentence):
    try:
        doc = nlp(str(sentence))
        return [(token.text, token.pos_) for token in doc]
    except Exception as e:
        logging.warning(f"Error in POS tagging for sentence: {str(e)}")
        return []


df["POS_Tags"] = df["Sentence"].apply(extract_pos_tags)


# (TF-IDF)
try:
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df["Sentence"].astype(str))
    df["TF_IDF"] = list(tfidf_matrix.toarray())
    logging.info("TF-IDF features extracted successfully")
except Exception as e:
    logging.error(f"Error in TF-IDF processing: {str(e)}")
    df["TF_IDF"] = [[] for _ in range(len(df))]


#Sentiment analysis
sentiment_analyzer = SentimentIntensityAnalyzer()


def get_sentiment_score(sentence):
    try:
        scores = sentiment_analyzer.polarity_scores(str(sentence))
        return scores["compound"]
    except Exception as e:
        logging.warning(f"Error in sentiment analysis: {str(e)}")
        return 0.0


df["Sentiment_Score"] = df["Sentence"].apply(get_sentiment_score)

# pretrained word embeddings
try:
    nlp_vec = spacy.load("en_core_web_md")
except Exception as e:
    logging.error(f"Error loading spaCy word vectors model: {str(e)}")
    raise


def get_pretrained_embedding(sentence):
    try:
        doc = nlp_vec(str(sentence))
        if doc.has_vector:
            return doc.vector
        return np.zeros(nlp_vec.vocab.vectors_length)
    except Exception as e:
        logging.warning(f"Error in pretrained embedding: {str(e)}")
        return np.zeros(nlp_vec.vocab.vectors_length)


df["Pretrained_Embeddings"] = df["Sentence"].apply(get_pretrained_embedding)



# custom word embedding model
def tokenize_sentence(sentence):
    try:
        return nltk.word_tokenize(str(sentence))
    except Exception as e:
        logging.warning(f"Error tokenizing sentence: {str(e)}")
        return []


try:
    #preparing tokenized sentences
    sentences_tokenized = df["Sentence"].apply(tokenize_sentence).tolist()

    # filtering out empty lists
    sentences_tokenized = [sent for sent in sentences_tokenized if sent]

    if not sentences_tokenized:
        raise ValueError("No valid sentences for Word2Vec training")

    # training Word2Vec model
    custom_model = Word2Vec(
        sentences=sentences_tokenized, vector_size=100, window=5, min_count=1, workers=4
    )
    logging.info("Custom Word2Vec model trained successfully")
except Exception as e:
    logging.error(f"Error in Word2Vec model training: {str(e)}")
    #creating a dummy model with zero vectors
    custom_model = type("DummyWord2Vec", (), {"wv": {}, "vector_size": 100})


def get_custom_embedding(sentence):
    try:
        tokens = tokenize_sentence(sentence)
        vectors = []
        for token in tokens:
            if token in custom_model.wv:
                vectors.append(custom_model.wv[token])
        if vectors:
            return np.mean(vectors, axis=0)
        return np.zeros(custom_model.vector_size)
    except Exception as e:
        logging.warning(f"Error in custom embedding: {str(e)}")
        return np.zeros(100)


df["Custom_Embeddings"] = df["Sentence"].apply(get_custom_embedding)



# additional NLP feature: named netity count
def count_named_entities(sentence):
    try:
        doc = nlp(str(sentence))
        return len(doc.ents)
    except Exception as e:
        logging.warning(f"Error counting entities: {str(e)}")
        return 0


df["Entity_Count"] = df["Sentence"].apply(count_named_entities)

# saving the Dataset
try:
    output_path = r"D:\YearTwoAI\Block C\2024-25c-fai2-adsai-VictoriaVicheva233182\Week 1\transcribed_data_with_features.csv"
    df.to_csv(output_path, index=False)
    logging.info("Data successfully saved to CSV")
except Exception as e:
    logging.error(f"Error saving data: {str(e)}")

logging.info("Processing complete. All features have been extracted.")

d:\anaconda\envs\tf_c\lib\site-packages\numpy\_core\_dtype.py:106: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if dtype.type == np.bool:


TypeError: 